# Исследовательский анализ данных: Deepfake Video Detection (DFDC02)
## Spatiotemporal Analysis Pipeline — Pre-modeling Stage

**ВКР:** «Метод выявления манипулированных видео на основе пространственно-временного анализа»

**Фокус данного EDA:** основной экспериментальный датасет DFDC_Dataset_02.

---

## 1. Configuration

In [29]:
# =====================================================================
# CONFIGURATION — PATHS FOR DFDC02 EDA
# =====================================================================

from pathlib import Path

# ─── Project root ───
PROJECT_ROOT = Path("/Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection")
assert PROJECT_ROOT.is_dir(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

# ─── Data paths ───
# Основной датасет для обучения модели: DFDC subset 02 (raw video)
DFDC_RAW_ROOT = PROJECT_ROOT / "data" / "DFDC_Dataset_02"

# Дополнительный датасет для cross-eval: DFD_01 (raw video)
DFD_RAW_ROOT = PROJECT_ROOT / "data" / "DFD_01"

# Preprocessed frame folders (если уже созданы)
PREPROCESSED_DFDC02 = PROJECT_ROOT / "preprocessed_data" / "preprocessed_DFDC02_16"
PREPROCESSED_DFD01  = PROJECT_ROOT / "preprocessed_data" / "preprocessed_DFD01_16"

# ─── Определяем что доступно ───
HAS_DFDC_RAW    = DFDC_RAW_ROOT.is_dir()
HAS_DFD_RAW     = DFD_RAW_ROOT.is_dir()
HAS_DFDC_PREPROCESSED = PREPROCESSED_DFDC02.is_dir()
HAS_DFD_PREPROCESSED  = PREPROCESSED_DFD01.is_dir()

print(f"DFDC_RAW_ROOT:        {DFDC_RAW_ROOT} — exists: {HAS_DFDC_RAW}")
print(f"DFD_RAW_ROOT:         {DFD_RAW_ROOT} — exists: {HAS_DFD_RAW}")
print(f"PREPROCESSED_DFDC02:  {PREPROCESSED_DFDC02} — exists: {HAS_DFDC_PREPROCESSED}")
print(f"PREPROCESSED_DFD01:   {PREPROCESSED_DFD01} — exists: {HAS_DFD_PREPROCESSED}")

# ─── Основной датасет для этого EDA ───
# Приоритет: preprocessed > raw
if HAS_DFDC_PREPROCESSED:
    PRIMARY_DATASET_ROOT = PREPROCESSED_DFDC02
    PRIMARY_DATASET_NAME = "DFDC02_preprocessed"
    PRIMARY_DATA_TYPE = "frames"
elif HAS_DFDC_RAW:
    PRIMARY_DATASET_ROOT = DFDC_RAW_ROOT
    PRIMARY_DATASET_NAME = "DFDC02_raw"
    PRIMARY_DATA_TYPE = "video"
else:
    raise FileNotFoundError("Ни raw, ни preprocessed DFDC02 не найдены!")

# Дополнительный датасет (опционально)
if HAS_DFD_PREPROCESSED:
    SECONDARY_DATASET_ROOT = PREPROCESSED_DFD01
    SECONDARY_DATASET_NAME = "DFD01_preprocessed"
elif HAS_DFD_RAW:
    SECONDARY_DATASET_ROOT = DFD_RAW_ROOT
    SECONDARY_DATASET_NAME = "DFD01_raw"
else:
    SECONDARY_DATASET_ROOT = None
    SECONDARY_DATASET_NAME = None

HAS_SECONDARY = SECONDARY_DATASET_ROOT is not None

print(f"\nPRIMARY:   {PRIMARY_DATASET_NAME} → {PRIMARY_DATASET_ROOT}")
if HAS_SECONDARY:
    print(f"SECONDARY: {SECONDARY_DATASET_NAME} → {SECONDARY_DATASET_ROOT}")
else:
    print("SECONDARY: не найден")

# ─── Report paths ───
REPORTS_DIR  = PROJECT_ROOT / "EDA" / "reports_final"
PLOTS_DIR    = REPORTS_DIR / "plots"
TABLES_DIR   = REPORTS_DIR / "tables"
INTERIM_DIR  = REPORTS_DIR / "interim"
CLEAN_DIR    = REPORTS_DIR / "clean"
MANIFESTS_DIR = REPORTS_DIR / "manifests"

print(f"\nREPORTS_DIR: {REPORTS_DIR}")

# ─── Analysis parameters ───
RANDOM_SEED = 42
SAVE_ARTIFACTS = True
RUN_HEAVY_ANALYSIS = True

# Параметры качества
MIN_RESOLUTION = 64
MIN_DURATION_SEC = 0.5
MIN_VALID_FACE_RATIO = 0.55
TARGET_CLIP_LEN = 16

# ─── Backward compatibility aliases (для остального кода) ───
# Эти переменные используются в остальных ячейках
CROPPED_ROOT = PRIMARY_DATASET_ROOT
VIDEO_ROOT = SECONDARY_DATASET_ROOT
HAS_VIDEO_DATASET = HAS_SECONDARY


DFDC_RAW_ROOT:        /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/data/DFDC_Dataset_02 — exists: True
DFD_RAW_ROOT:         /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/data/DFD_01 — exists: True
PREPROCESSED_DFDC02:  /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/preprocessed_data/preprocessed_DFDC02_16 — exists: False
PREPROCESSED_DFD01:   /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/preprocessed_data/preprocessed_DFD01_16 — exists: False

PRIMARY:   DFDC02_raw → /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/data/DFDC_Dataset_02
SECONDARY: DFD01_raw → /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/data/DFD_01

REPORTS_DIR: /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/EDA/reports_final


In [31]:
import os, sys, hashlib, warnings, time, json, re, gc
from pathlib import Path
from collections import Counter, defaultdict, OrderedDict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "figure.dpi": 110,
    "savefig.bbox": "tight",
    "savefig.dpi": 150,
})

for d in [REPORTS_DIR, PLOTS_DIR, TABLES_DIR, INTERIM_DIR, CLEAN_DIR, MANIFESTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

try:
    import cv2
    print(f"OpenCV: {cv2.__version__}")
except ImportError:
    raise ImportError("Install OpenCV: python -m pip install opencv-python")

print(f"Python executable: {sys.executable}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PRIMARY (CROPPED_ROOT alias): {CROPPED_ROOT} — exists: {Path(CROPPED_ROOT).is_dir()}")
print(f"SECONDARY (VIDEO_ROOT alias): {VIDEO_ROOT} — exists: {Path(VIDEO_ROOT).is_dir() if VIDEO_ROOT else False}")
print(f"REPORTS_DIR:  {REPORTS_DIR}")

OpenCV: 4.11.0
Python executable: /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/.venv/bin/python
NumPy: 1.26.4
Pandas: 2.2.3
PRIMARY (CROPPED_ROOT alias): /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/data/DFDC_Dataset_02 — exists: True
SECONDARY (VIDEO_ROOT alias): /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/data/DFD_01 — exists: True
REPORTS_DIR:  /Users/alext/Desktop/MIFI 3/VKR_Final/Final_Project_deefake_detection/EDA/reports_final


In [33]:
# =====================================================================
# UTILITY FUNCTIONS
# =====================================================================

def save_fig(name: str, fig=None, subdir=PLOTS_DIR):
    if not SAVE_ARTIFACTS:
        return
    out_path = Path(subdir) / f"{name}.png"
    (fig or plt).savefig(out_path, bbox_inches="tight", dpi=150)

def save_csv(df, name: str, subdir=TABLES_DIR, index: bool = False):
    if not SAVE_ARTIFACTS:
        return
    out_path = Path(subdir) / name
    Path(subdir).mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=index)

def mini_report(title: str, items: dict):
    print(f"\n{'━' * 60}")
    print(f"  {title}")
    print(f"{'━' * 60}")
    for k, v in items.items():
        print(f"  {k}: {v}")
    print(f"{'━' * 60}\n")

def infer_label(path_str: str) -> str:
    low = str(path_str).lower()

    fake_keywords = [
        "fake", "manipulated", "altered", "deepfake", "swap",
        "forged", "synthetic", "tampered"
    ]
    real_keywords = [
        "real", "original", "pristine", "authentic",
        "genuine", "clean"
    ]

    for kw in fake_keywords:
        if kw in low:
            return "fake"
    for kw in real_keywords:
        if kw in low:
            return "real"
    return "unknown"

def infer_label_int(label: str) -> Optional[int]:
    return LABEL_MAP.get(str(label).lower(), -1)

def infer_source_dataset(path_str: str) -> str:
    low = str(path_str).lower()

    if "dfdc_dataset_02" in low:
        return "DFDC02"
    if "preprocessed_dfd01" in low:
        return "DFD01"
    return "unknown"

def infer_file_type(path_str: str) -> str:
    ext = Path(path_str).suffix.lower()
    if ext in SUPPORTED_VIDEO_EXTS:
        return "video"
    if ext in SUPPORTED_IMAGE_EXTS:
        return "image"
    return "unsupported"

def infer_group_id(path_str: str) -> str:
    path = Path(path_str)
    ext = path.suffix.lower()

    if ext in SUPPORTED_VIDEO_EXTS:
        return path.stem

    if ext in SUPPORTED_IMAGE_EXTS:
        if path.parent and path.parent.name:
            return path.parent.name
        return path.stem

    return path.stem

def infer_parent_id(path_str: str) -> Optional[str]:
    path = Path(path_str)
    if path.parent and path.parent.name:
        return path.parent.name
    return None

def fast_hash(path: str, n_bytes: int = HASH_SAMPLE_BYTES) -> Optional[str]:
    try:
        h = hashlib.md5()
        with open(path, "rb") as f:
            h.update(f.read(n_bytes))
        return h.hexdigest()
    except Exception:
        return None

## 2. Data Contract

Определяем обязательные колонки master metadata table. Каждый sample в итоговой таблице должен иметь эти поля.

In [36]:
DATA_CONTRACT_COLUMNS = [
    "sample_id",
    "path",
    "source_dataset",
    "file_type",
    "label",
    "label_int",
    "ext",
    "size_bytes",
    "is_readable",
    "read_error",
    "file_hash",
    "group_id",
    "parent_id",
    "width",
    "height",
    "fps",
    "frame_count",
    "duration_sec",
    "codec",
    "aspect_ratio",
    "issue_flags",
]

print(f"Data contract: {len(DATA_CONTRACT_COLUMNS)} обязательных колонок")
print(DATA_CONTRACT_COLUMNS)

Data contract: 21 обязательных колонок
['sample_id', 'path', 'source_dataset', 'file_type', 'label', 'label_int', 'ext', 'size_bytes', 'is_readable', 'read_error', 'file_hash', 'group_id', 'parent_id', 'width', 'height', 'fps', 'frame_count', 'duration_sec', 'codec', 'aspect_ratio', 'issue_flags']


## 3. Full Raw Data Inventory

Рекурсивное сканирование обоих датасетов. Для каждого файла: metadata + readability check + hash.

In [39]:
def get_video_meta(path: str) -> dict:
    """Извлекает базовые метаданные видео через OpenCV."""
    m = {
        "is_readable": False,
        "read_error": None,
        "width": 0,
        "height": 0,
        "fps": 0.0,
        "frame_count": 0,
        "duration_sec": 0.0,
        "codec": "",
    }

    cap = None
    try:
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            m["read_error"] = "cannot_open"
            return m

        m["is_readable"] = True
        m["width"] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        m["height"] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        m["fps"] = round(float(cap.get(cv2.CAP_PROP_FPS) or 0.0), 2)
        m["frame_count"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

        fourcc = int(cap.get(cv2.CAP_PROP_FOURCC) or 0)
        m["codec"] = "".join(chr((fourcc >> (8 * i)) & 0xFF) for i in range(4)).strip("\x00")

        if m["fps"] > 0 and m["frame_count"] > 0:
            m["duration_sec"] = round(m["frame_count"] / m["fps"], 2)

    except Exception as e:
        m["read_error"] = str(e)[:200]
    finally:
        if cap is not None:
            cap.release()

    return m


def get_image_meta(path: str) -> dict:
    """Извлекает базовые метаданные изображения через OpenCV."""
    m = {
        "is_readable": False,
        "read_error": None,
        "width": 0,
        "height": 0,
        "fps": 0.0,
        "frame_count": 1,
        "duration_sec": 0.0,
        "codec": "",
    }

    try:
        img = cv2.imread(str(path))
        if img is None:
            m["read_error"] = "cannot_read"
            return m

        m["is_readable"] = True
        m["height"], m["width"] = img.shape[:2]

    except Exception as e:
        m["read_error"] = str(e)[:200]

    return m


def scan_dataset(root: str, dataset_name: str):
    """Сканирует датасет. Возвращает (records, errors, unsupported)."""
    records, errors, unsupported = [], [], []

    root_p = Path(root)
    if not root_p.exists():
        print(f"[SKIP] Path does not exist: {root_p}")
        return records, errors, unsupported

    all_files = sorted([p for p in root_p.rglob("*") if p.is_file()])
    print(f"Сканирование {dataset_name}: {len(all_files)} файлов...")

    for idx, fpath in enumerate(tqdm(all_files, desc=dataset_name)):
        ext = fpath.suffix.lower()
        rel = str(fpath.relative_to(root_p))

        if ext not in SUPPORTED_IMAGE_EXTS and ext not in SUPPORTED_VIDEO_EXTS:
            unsupported.append({
                "path": str(fpath),
                "ext": ext,
                "dataset": dataset_name,
            })
            continue

        file_type = infer_file_type(str(fpath))
        size_b = fpath.stat().st_size if fpath.exists() else 0

        if file_type == "video":
            meta = get_video_meta(str(fpath))
        elif file_type == "image":
            meta = get_image_meta(str(fpath))
        else:
            unsupported.append({
                "path": str(fpath),
                "ext": ext,
                "dataset": dataset_name,
            })
            continue

        label = infer_label(rel)
        label_int = infer_label_int(label)
        group_id = infer_group_id(str(fpath))
        parent_id = infer_parent_id(str(fpath))
        fhash = fast_hash(str(fpath))

        width = int(meta["width"] or 0)
        height = int(meta["height"] or 0)
        aspect_ratio = round(width / max(height, 1), 3) if height > 0 else 0.0

        rec = {
            "sample_id": f"{dataset_name}_{idx:06d}",
            "path": str(fpath),
            "source_dataset": dataset_name,
            "file_type": file_type,
            "label": label,
            "label_int": label_int,
            "ext": ext,
            "size_bytes": size_b,
            "is_readable": meta["is_readable"],
            "read_error": meta["read_error"],
            "file_hash": fhash,
            "group_id": group_id,
            "parent_id": parent_id,
            "width": width,
            "height": height,
            "fps": meta["fps"],
            "frame_count": meta["frame_count"],
            "duration_sec": meta["duration_sec"],
            "codec": meta["codec"],
            "aspect_ratio": aspect_ratio,
            "issue_flags": "",
        }
        records.append(rec)

        if meta["read_error"]:
            errors.append({
                "path": str(fpath),
                "error": meta["read_error"],
                "dataset": dataset_name,
            })

    return records, errors, unsupported

In [41]:
# ═══════════════════════════════════════════════════════════════
# Сканируем датасеты
# ═══════════════════════════════════════════════════════════════

# Основной датасет: DFDC02
rec_primary, err_primary, uns_primary = scan_dataset(
    PRIMARY_DATASET_ROOT, PRIMARY_DATASET_NAME
)

# Дополнительный датасет (если есть)
if HAS_SECONDARY:
    rec_secondary, err_secondary, uns_secondary = scan_dataset(
        SECONDARY_DATASET_ROOT, SECONDARY_DATASET_NAME
    )
else:
    rec_secondary, err_secondary, uns_secondary = [], [], []

all_records = rec_primary + rec_secondary
all_errors = err_primary + err_secondary
all_unsupported = uns_primary + uns_secondary

df = pd.DataFrame(all_records)
df_errors = pd.DataFrame(all_errors) if all_errors else pd.DataFrame(columns=["path", "error", "dataset"])
df_unsupported = pd.DataFrame(all_unsupported) if all_unsupported else pd.DataFrame(columns=["path", "ext", "dataset"])

save_csv(df, "master_raw_inventory.csv")
save_csv(df_errors, "inventory_errors.csv")
save_csv(df_unsupported, "unsupported_files.csv")

mini_report("RAW INVENTORY", {
    "Всего samples": len(df),
    "Images": int((df["file_type"] == "image").sum()) if not df.empty else 0,
    "Videos": int((df["file_type"] == "video").sum()) if not df.empty else 0,
    "Real": int((df["label"] == "real").sum()) if not df.empty else 0,
    "Fake": int((df["label"] == "fake").sum()) if not df.empty else 0,
    "Unknown label": int((df["label"] == "unknown").sum()) if not df.empty else 0,
    "Unreadable": int((~df["is_readable"]).sum()) if not df.empty else 0,
    "Read errors": len(df_errors),
    "Unsupported files": len(df_unsupported),
})

# Breakdown по датасетам
if not df.empty:
    print("\n--- По датасетам ---")
    for ds in df["source_dataset"].unique():
        sub = df[df["source_dataset"] == ds]
        print(f"  {ds}: {len(sub)} files "
              f"(real={int((sub['label']=='real').sum())}, "
              f"fake={int((sub['label']=='fake').sum())}, "
              f"images={int((sub['file_type']=='image').sum())}, "
              f"videos={int((sub['file_type']=='video').sum())})")


Сканирование DFDC02_raw: 3294 файлов...


DFDC02_raw:   0%|                             | 1/3294 [00:00<00:06, 539.39it/s]


NameError: name 'LABEL_MAP' is not defined

In [ ]:
# Визуализация инвентаризации.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. По датасету и типу.
ct1 = df.groupby(['source_dataset', 'file_type']).size().unstack(fill_value=0)
ct1.plot.bar(ax=axes[0], color=['#3498db', '#e67e22'])
axes[0].set_title('Файлы: датасет × тип')
axes[0].set_ylabel('Количество')
axes[0].tick_params(axis='x', rotation=0)

# 2. По датасету и label.
ct2 = df.groupby(['source_dataset', 'label']).size().unstack(fill_value=0)
ct2.plot.bar(ax=axes[1], color=['#e74c3c', '#2ecc71', '#95a5a6'])
axes[1].set_title('Файлы: датасет × label')
axes[1].tick_params(axis='x', rotation=0)

# 3. Readable.
ct3 = df.groupby(['source_dataset', 'is_readable']).size().unstack(fill_value=0)
ct3.plot.bar(ax=axes[2], stacked=True, color=['#e74c3c', '#2ecc71'])
axes[2].set_title('Читаемость по датасетам')
axes[2].legend(['Unreadable', 'Readable'])
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
save_fig("01_raw_inventory")
plt.show()

# Cross-tab.
xt = pd.crosstab(df['source_dataset'], df['label'], margins=True)
save_csv(xt.reset_index(), "dataset_label_crosstab.csv")
print(xt)

## 4. Raw EDA

Распределения ключевых характеристик: длительность, fps, разрешение, размер файла, баланс классов.

In [ ]:
# === 4.1. Видео EDA ===
dfv = df[(df['file_type']=='video') & (df['is_readable'])].copy()

if len(dfv) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))

    # Duration by label.
    for lab, c in [('real','#2ecc71'),('fake','#e74c3c')]:
        s = dfv[dfv['label']==lab]['duration_sec']
        if len(s) > 0: axes[0,0].hist(s, bins=30, alpha=0.6, label=lab, color=c)
    axes[0,0].set_title('Длительность видео (сек)'); axes[0,0].legend()

    # FPS.
    dfv['fps'].hist(ax=axes[0,1], bins=25, color='#3498db', edgecolor='k')
    axes[0,1].set_title('FPS')

    # Resolution width.
    dfv['width'].hist(ax=axes[0,2], bins=25, color='#9b59b6', edgecolor='k')
    axes[0,2].set_title('Width (px)')

    # Frame count by label.
    for lab, c in [('real','#2ecc71'),('fake','#e74c3c')]:
        s = dfv[dfv['label']==lab]['frame_count']
        if len(s) > 0: axes[1,0].hist(s, bins=30, alpha=0.6, label=lab, color=c)
    axes[1,0].set_title('Frame count'); axes[1,0].legend()

    # File size boxplot.
    dfv['size_mb'] = dfv['size_bytes'] / 1e6
    sns.boxplot(data=dfv, x='label', y='size_mb', ax=axes[1,1],
                palette={'real':'#2ecc71','fake':'#e74c3c'})
    axes[1,1].set_title('File size (MB)')

    # Aspect ratio.
    dfv['aspect_ratio'].hist(ax=axes[1,2], bins=25, color='#e67e22', edgecolor='k')
    axes[1,2].set_title('Aspect ratio')

    plt.suptitle('EDA: DFD_Original (видео)', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_fig("02_video_eda")
    plt.show()

    # Summary stats.
    stats_v = dfv[['duration_sec','fps','width','height','frame_count','size_bytes']].describe().T
    stats_v.columns = ['count','mean','std','min','25%','50%','75%','max']
    save_csv(stats_v.reset_index().rename(columns={'index':'metric'}), "summary_video_stats.csv")
    print(stats_v.round(1))
else:
    print("Видео не найдены — пропуск.")

In [ ]:
# === 4.2. Изображения EDA ===
dfi = df[(df['file_type']=='image') & (df['is_readable'])].copy()

if len(dfi) > 0:
    dfi['size_mb'] = dfi['size_bytes'] / 1e6

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    # Class balance.
    dfi['label'].value_counts().plot.bar(ax=axes[0], color=['#e74c3c','#2ecc71'])
    axes[0].set_title('Class balance (cropped)')

    # Width dist.
    for lab, c in [('real','#2ecc71'),('fake','#e74c3c')]:
        s = dfi[dfi['label']==lab]['width']
        if len(s) > 0: axes[1].hist(s, bins=30, alpha=0.6, label=lab, color=c)
    axes[1].set_title('Width distribution'); axes[1].legend()

    # File size.
    sns.boxplot(data=dfi, x='label', y='size_mb', ax=axes[2],
                palette={'real':'#2ecc71','fake':'#e74c3c'})
    axes[2].set_title('File size (MB)')

    # Aspect ratio.
    dfi['aspect_ratio'].hist(ax=axes[3], bins=30, color='#e67e22', edgecolor='k')
    axes[3].set_title('Aspect ratio')

    plt.suptitle('EDA: DFDC_Cropped (изображения)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_fig("03_cropped_eda")
    plt.show()

    stats_i = dfi[['width','height','size_bytes','aspect_ratio']].describe().T
    save_csv(stats_i.reset_index().rename(columns={'index':'metric'}), "summary_image_stats.csv")
    print(stats_i.round(1))

    mini_report("RAW EDA", {
        "Videos readable": len(dfv) if 'dfv' in dir() else 0,
        "Images readable": len(dfi),
        "Median video duration": f"{dfv['duration_sec'].median():.1f}s" if len(dfv)>0 else "N/A",
        "Median image width": f"{dfi['width'].median():.0f}px",
        "Class balance (cropped)": dict(dfi['label'].value_counts()),
    })
else:
    print("Изображения не найдены — пропуск.")

## 5. Data Quality Audit

Систематическая проверка каждого sample на проблемы. Результат: issue_flags для каждого sample + problem_registry.

In [ ]:
def audit_sample(row: pd.Series) -> List[str]:
    """Возвращает список проблем (issue flags) для одного sample."""
    flags = []

    if not row['is_readable']:
        flags.append('unreadable')

    if row['size_bytes'] == 0:
        flags.append('empty_file')

    if row['size_bytes'] < 1024:
        flags.append('tiny_file')

    if row['file_type'] == 'video':
        if row['duration_sec'] <= 0:
            flags.append('zero_duration')
        elif row['duration_sec'] < MIN_DURATION_SEC:
            flags.append('too_short')

        if row['fps'] <= 0 or np.isnan(row['fps']) or np.isinf(row['fps']):
            flags.append('invalid_fps')
        elif row['fps'] < VALID_FPS_RANGE[0] or row['fps'] > VALID_FPS_RANGE[1]:
            flags.append('suspicious_fps')

        if row['frame_count'] < TARGET_CLIP_LEN:
            flags.append('too_few_frames')

    if row['is_readable']:
        if min(row['width'], row['height']) < MIN_RESOLUTION:
            flags.append('low_resolution')

        ar = row['aspect_ratio']
        if ar > 0 and (ar < 0.25 or ar > 4.0):
            flags.append('extreme_aspect_ratio')

    if row['label'] == 'unknown':
        flags.append('missing_label')

    return flags

# Применяем ко всем samples.
print("Аудит качества данных...")
issue_lists = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Quality audit"):
    flags = audit_sample(row)
    issue_lists.append('|'.join(flags) if flags else '')

df['issue_flags'] = issue_lists
df['has_issues'] = df['issue_flags'].str.len() > 0
df['issue_count'] = df['issue_flags'].apply(lambda x: len(x.split('|')) if x else 0)

print(f"\nSamples с проблемами: {df['has_issues'].sum()} / {len(df)} "
      f"({df['has_issues'].mean()*100:.1f}%)")

In [ ]:
# Проверка дубликатов по hash
print("Проверка hash-дубликатов...")

hash_counts = df.loc[df["file_hash"].notna(), "file_hash"].value_counts()
dup_hashes = hash_counts[hash_counts > 1]

n_dup_hash_groups = int(len(dup_hashes))
n_dup_hash_files = int(df["file_hash"].isin(dup_hashes.index).sum())
n_dup_hash_extra = int(n_dup_hash_files - n_dup_hash_groups)

print(f"Групп hash-дубликатов: {n_dup_hash_groups}")
print(f"Файлов в duplicate-hash группах: {n_dup_hash_files}")
print(f'Лишних копий: {n_dup_hash_extra}')

# Помечаем duplicate_hash у всех повторов, кроме первого
dup_hash_mask = df.duplicated(subset=["file_hash"], keep="first") & df["file_hash"].notna()
df.loc[dup_hash_mask, "issue_flags"] = df.loc[dup_hash_mask, "issue_flags"].apply(
    lambda x: f"{x}|duplicate_hash" if x else "duplicate_hash"
)

# Проверка дубликатов по имени файла внутри одного датасета
print("Проверка filename-дубликатов...")
df["basename"] = df["path"].apply(lambda x: Path(x).name)

fname_dup_groups = (
    df.groupby(["source_dataset", "basename"])
      .size()
      .reset_index(name="count")
)
fname_dup_groups = fname_dup_groups[fname_dup_groups["count"] > 1]

n_fname_dup_groups = int(len(fname_dup_groups))
n_fname_dup_files = int(
    df.merge(
        fname_dup_groups[["source_dataset", "basename"]],
        on=["source_dataset", "basename"],
        how="inner"
    ).shape[0]
)

print(f"Групп filename-дубликатов: {n_fname_dup_groups}")
print(f"Файлов в filename-duplicate группах: {n_fname_dup_files}")

# Помечаем duplicate_name у всех повторов, кроме первого
dup_name_mask = df.duplicated(subset=["source_dataset", "basename"], keep="first")
df.loc[dup_name_mask, "issue_flags"] = df.loc[dup_name_mask, "issue_flags"].apply(
    lambda x: f"{x}|duplicate_name" if x else "duplicate_name"
)

In [ ]:
# === Problem Registry ===
all_flags = []
for _, row in df[df["issue_flags"].fillna("").str.len() > 0].iterrows():
    for flag in row["issue_flags"].split("|"):
        flag = flag.strip()
        if not flag:
            continue
        all_flags.append({
            "sample_id": row["sample_id"],
            "path": row["path"],
            "source_dataset": row["source_dataset"],
            "label": row["label"],
            "issue": flag,
        })

df_problems = pd.DataFrame(all_flags) if all_flags else pd.DataFrame(
    columns=["sample_id", "path", "source_dataset", "label", "issue"]
)

save_csv(df_problems, "problem_registry.csv")

issue_summary = pd.DataFrame(columns=["count", "datasets"])

if len(df_problems) > 0:
    issue_summary = (
        df_problems.groupby("issue")
        .agg(
            count=("sample_id", "count"),
            datasets=("source_dataset", lambda x: ", ".join(sorted(x.unique()))),
        )
        .sort_values("count", ascending=False)
    )
    save_csv(issue_summary.reset_index(), "quality_issue_summary.csv")
    print(issue_summary)

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # 1. Count по типу проблемы
    issue_summary["count"].plot.barh(ax=axes[0], color="#e74c3c")
    axes[0].set_title("Количество по типу проблемы")
    axes[0].invert_yaxis()

    # 2. Heatmap issue × dataset
    xt_ds = pd.crosstab(df_problems["issue"], df_problems["source_dataset"])
    sns.heatmap(xt_ds, annot=True, fmt="d", cmap="OrRd", ax=axes[1])
    axes[1].set_title("Проблемы × датасет")

    # 3. Heatmap issue × label
    xt_lb = pd.crosstab(df_problems["issue"], df_problems["label"])
    sns.heatmap(xt_lb, annot=True, fmt="d", cmap="OrRd", ax=axes[2])
    axes[2].set_title("Проблемы × label")

    plt.tight_layout()
    save_fig("04_quality_audit")
    plt.show()
else:
    print("Проблем не обнаружено.")

mini_report("QUALITY AUDIT", {
    "Samples с проблемами": f"{int(df['has_issues'].sum())} / {len(df)}",
    "Типов проблем": int(df_problems["issue"].nunique()) if len(df_problems) > 0 else 0,
    "Hash-дубликат групп": n_dup_hash_groups,
    "Лишних hash-копий": n_dup_hash_extra,
    "Top проблема": issue_summary.index[0] if len(issue_summary) > 0 else "нет",
})

## 6. Leakage Risk Analysis

Определяем group_id для leakage-safe split. Проверяем: нет ли одного и того же источника в разных потенциальных split'ах.

In [ ]:
print("=== LEAKAGE RISK ANALYSIS ===\n")

# 1. Анализ group_id
print(f"Уникальных group_id: {df['group_id'].nunique()}")
group_sizes = df.groupby("group_id").size()
print(
    f"Размеры групп: min={group_sizes.min()}, "
    f"median={group_sizes.median():.0f}, max={group_sizes.max()}"
)

# 2. Проверка: есть ли группы с обоими labels
group_labels = df.groupby("group_id")["label"].nunique()
mixed_groups = group_labels[group_labels > 1]
print(f"\nГруппы с разными labels: {len(mixed_groups)}")

if len(mixed_groups) > 0 and len(mixed_groups) <= 10:
    for gid in mixed_groups.index:
        sub = df[df["group_id"] == gid]
        print(f"  {gid}: {dict(sub['label'].value_counts())}")

# 3. Проверка potential parent-child / derived relations внутри raw video dataset
leakage_candidates = []

dfv_only = df[df["source_dataset"] == PRIMARY_DATASET_NAME].copy()
if len(dfv_only) > 0:
    real_stems = sorted(set(dfv_only[dfv_only["label"] == "real"]["group_id"]))
    fake_stems = sorted(set(dfv_only[dfv_only["label"] == "fake"]["group_id"]))

    for rs in real_stems[:300]:
        for fs in fake_stems[:300]:
            if rs == fs:
                leakage_candidates.append({
                    "candidate_type": "exact_group_match",
                    "real_id": rs,
                    "fake_id": fs,
                })
            elif rs in fs or fs in rs:
                leakage_candidates.append({
                    "candidate_type": "substring_match",
                    "real_id": rs,
                    "fake_id": fs,
                })

df_leakage_candidates = pd.DataFrame(leakage_candidates)
save_csv(df_leakage_candidates, "leakage_candidates.csv")

print(f"\nПотенциальные leakage-кандидаты: {len(df_leakage_candidates)}")
if len(df_leakage_candidates) > 0:
    print("  → Эти пары нельзя разносить по разным split.")

# 4. Пересечение group_id между датасетами
groups_frames = set(df[df["source_dataset"] == SECONDARY_DATASET_NAME if HAS_SECONDARY else "none"]["group_id"])
groups_video = set(df[df["source_dataset"] == PRIMARY_DATASET_NAME]["group_id"])
cross_overlap = groups_frames & groups_video

print(f"\nПересечение group_id между датасетами: {len(cross_overlap)}")
if len(cross_overlap) > 0 and len(cross_overlap) <= 10:
    print(list(cross_overlap)[:10])

# 5. Leakage groups summary
leakage_groups = (
    df.groupby("group_id")
      .agg(
          count=("sample_id", "count"),
          labels=("label", lambda x: "|".join(sorted(set(x)))),
          datasets=("source_dataset", lambda x: "|".join(sorted(set(x)))),
      )
      .reset_index()
      .sort_values(["count", "group_id"], ascending=[False, True])
)

save_csv(leakage_groups, "leakage_groups.csv")

mini_report("LEAKAGE AUDIT", {
    "Уникальных group_id": int(df["group_id"].nunique()),
    "Mixed-label groups": int(len(mixed_groups)),
    "Leakage candidates": int(len(df_leakage_candidates)),
    "Cross-dataset group overlap": int(len(cross_overlap)),
    "Split key": "group_id",
})

print("\n--- РЕШЕНИЕ ---")
print("Split key: group_id")
print("Все samples с одним group_id → строго в один split.")
print("DFD_Frames использовать как основной training dataset.")
print("DFDC_Video использовать отдельно для raw video audit / cross-check.")

## 7. Frame-level Analysis

Извлекаем кадры, визуализируем примеры real/fake, выявляем проблемные случаи.

In [ ]:
def extract_frames_uniform(video_path: str, n: int = 5) -> List[np.ndarray]:
    """Извлекает n кадров из видео (uniform sampling). Возвращает RGB frames."""
    frames = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return frames
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return frames
    indices = np.linspace(0, total-1, n, dtype=int)
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def load_image_rgb(path: str) -> Optional[np.ndarray]:
    img = cv2.imread(path)
    if img is not None:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return None

def show_grid(images, titles, suptitle, ncols=5, figsize=(20,8)):
    n = len(images)
    nrows = max(1, (n + ncols - 1) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n and images[i] is not None:
            ax.imshow(images[i])
            ax.set_title(titles[i] if i < len(titles) else '', fontsize=8)
        ax.axis('off')
    plt.suptitle(suptitle, fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig

In [ ]:
# === 7.1. Примеры из DFDC_Cropped ===
dfi_ok = df[(df['source_dataset']==PRIMARY_DATASET_NAME) & df['is_readable']].copy()

for label in ['real', 'fake']:
    subset = dfi_ok[dfi_ok['label']==label]
    if len(subset) < 5:
        continue
    sample_paths = subset.sample(min(10, len(subset)), random_state=RANDOM_SEED)['path'].tolist()
    imgs = [load_image_rgb(p) for p in sample_paths]
    titles = [Path(p).name[:20] for p in sample_paths]
    fig = show_grid(imgs, titles, f"Primary dataset: '{label}' examples")
    save_fig(f"05_cropped_{label}_examples")
    plt.show()

In [ ]:
# === 7.2. Кадры из DFD_Original (видео) ===
dfv_ok = df[(df['source_dataset']==SECONDARY_DATASET_NAME if HAS_SECONDARY else "__skip__") & df['is_readable'] &
            (df['file_type']=='video')].copy()

if len(dfv_ok) > 0:
    for label in ['real', 'fake']:
        subset = dfv_ok[dfv_ok['label']==label]
        if len(subset) < 2:
            continue
        sample_vids = subset.sample(min(3, len(subset)), random_state=RANDOM_SEED)
        all_frames, all_titles = [], []
        for _, row in sample_vids.iterrows():
            frames = extract_frames_uniform(row['path'], n=5)
            stem = Path(row['path']).stem[:12]
            for i, f in enumerate(frames):
                all_frames.append(f)
                all_titles.append(f"{stem}_f{i}")
        if all_frames:
            fig = show_grid(all_frames, all_titles,
                          f"Secondary dataset: '{label}' video frames", ncols=5)
            save_fig(f"06_video_{label}_frames")
            plt.show()

# Сохраняем frame sample metadata (для последующих секций).
frame_meta_records = []
sample_for_frames = df[df['is_readable']].sample(
    min(MAX_VIDEOS_FOR_HEAVY_ANALYSIS, len(df[df['is_readable']])),
    random_state=RANDOM_SEED
)

for _, row in tqdm(sample_for_frames.iterrows(), total=len(sample_for_frames),
                   desc="Frame sampling"):
    if row['file_type'] == 'video':
        frames = extract_frames_uniform(row['path'], n=MAX_FRAMES_PER_VIDEO_FOR_ANALYSIS)
    else:
        img = load_image_rgb(row['path'])
        frames = [img] if img is not None else []

    for fi, frame in enumerate(frames):
        frame_meta_records.append({
            'sample_id': row['sample_id'],
            'frame_idx': fi,
            'label': row['label'],
            'source_dataset': row['source_dataset'],
            'height': frame.shape[0],
            'width': frame.shape[1],
            'path': row['path'],
        })

df_frame_meta = pd.DataFrame(frame_meta_records)
save_csv(df_frame_meta, "frame_samples_metadata.csv")
print(f"\nSampled frames: {len(df_frame_meta)} из {len(sample_for_frames)} samples")

## 8. Face-level Analysis

Face detection на sampled frames. Для каждого кадра: число лиц, bbox, face_area_ratio.
Для каждого видео: face_detection_ratio, mean_face_area_ratio.

In [ ]:
# =====================================================================
# 8. FACE-LEVEL ANALYSIS
# =====================================================================

if RUN_HEAVY_ANALYSIS:
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )

    def detect_faces(image_rgb: np.ndarray) -> List[dict]:
        """Детекция лиц. Возвращает список bbox + area_ratio."""
        if image_rgb is None:
            return []

        gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
        img_h, img_w = gray.shape[:2]
        img_area = max(img_h * img_w, 1)

        rects = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(20, 20),
        )

        faces = []
        if isinstance(rects, np.ndarray) and len(rects) > 0:
            for (x, y, w, h) in rects:
                faces.append({
                    "x": int(x),
                    "y": int(y),
                    "w": int(w),
                    "h": int(h),
                    "area_ratio": round((w * h) / img_area, 6),
                })
        return faces

    print("Face detection on sampled frames...")

    face_records = []
    failed_face_cases = []

    readable_df = df[df["is_readable"]].copy()
    sample_for_face = readable_df.sample(
        min(MAX_VIDEOS_FOR_HEAVY_ANALYSIS, len(readable_df)),
        random_state=RANDOM_SEED
    )

    for _, row in tqdm(sample_for_face.iterrows(), total=len(sample_for_face), desc="Face detection"):
        if row["file_type"] == "video":
            frames = extract_frames_uniform(row["path"], n=MAX_FRAMES_PER_VIDEO_FOR_ANALYSIS)
        else:
            img = load_image_rgb(row["path"])
            frames = [img] if img is not None else []

        if not frames:
            failed_face_cases.append({
                "sample_id": row["sample_id"],
                "path": row["path"],
                "source_dataset": row["source_dataset"],
                "label": row["label"],
                "reason": "no_frames_loaded",
            })
            continue

        for fi, frame in enumerate(frames):
            if frame is None:
                failed_face_cases.append({
                    "sample_id": row["sample_id"],
                    "path": row["path"],
                    "source_dataset": row["source_dataset"],
                    "label": row["label"],
                    "reason": f"frame_{fi}_is_none",
                })
                continue

            faces = detect_faces(frame)
            n_faces = len(faces)
            largest_face_area = max([f["area_ratio"] for f in faces], default=0.0)

            if n_faces == 0:
                failed_face_cases.append({
                    "sample_id": row["sample_id"],
                    "path": row["path"],
                    "source_dataset": row["source_dataset"],
                    "label": row["label"],
                    "reason": "no_face_detected",
                    "frame_idx": fi,
                })

            for face_idx, f in enumerate(faces if faces else [{}]):
                face_records.append({
                    "sample_id": row["sample_id"],
                    "frame_idx": fi,
                    "face_idx": face_idx if n_faces > 0 else -1,
                    "path": row["path"],
                    "label": row["label"],
                    "source_dataset": row["source_dataset"],
                    "n_faces": n_faces,
                    "face_detected": n_faces > 0,
                    "multi_face": n_faces > 1,
                    "largest_face_area_ratio": largest_face_area,
                    "x": f.get("x", None),
                    "y": f.get("y", None),
                    "w": f.get("w", None),
                    "h": f.get("h", None),
                    "face_area_ratio": f.get("area_ratio", 0.0),
                })

    df_faces = pd.DataFrame(face_records)
    df_failed_faces = pd.DataFrame(failed_face_cases)

    save_csv(df_faces, "face_detections.csv")
    save_csv(df_failed_faces, "failed_face_cases.csv")

    if len(df_faces) > 0:
        face_video_summary = (
            df_faces.groupby("sample_id")
            .agg(
                path=("path", "first"),
                label=("label", "first"),
                source_dataset=("source_dataset", "first"),
                sampled_frames=("frame_idx", "nunique"),
                frames_with_face=("face_detected", lambda x: int(x.sum())),
                face_detection_ratio=("face_detected", lambda x: round(float(np.mean(x)), 4)),
                mean_face_area_ratio=("largest_face_area_ratio", lambda x: round(float(np.mean(x)), 6)),
                max_face_area_ratio=("largest_face_area_ratio", lambda x: round(float(np.max(x)), 6)),
                multi_face_ratio=("multi_face", lambda x: round(float(np.mean(x)), 4)),
                no_face_ratio=("face_detected", lambda x: round(float(1.0 - np.mean(x)), 4)),
            )
            .reset_index()
        )
    else:
        face_video_summary = pd.DataFrame(columns=[
            "sample_id", "path", "label", "source_dataset", "sampled_frames",
            "frames_with_face", "face_detection_ratio", "mean_face_area_ratio",
            "max_face_area_ratio", "multi_face_ratio", "no_face_ratio"
        ])

    save_csv(face_video_summary, "face_video_summary.csv")

    print(f"Face detections: {len(df_faces)} rows")
    print(f"Face video summary: {len(face_video_summary)} samples")
    print(f"Failed face cases: {len(df_failed_faces)}")

else:
    print("Face analysis skipped (RUN_HEAVY_ANALYSIS=False)")
    df_faces = pd.DataFrame()
    df_failed_faces = pd.DataFrame()
    face_video_summary = pd.DataFrame()

In [ ]:
# === Video-level face summary / visualization ===
if len(face_video_summary) > 0:
    failed_face_summary = face_video_summary[
        face_video_summary["face_detection_ratio"] < MIN_VALID_FACE_RATIO
    ].copy()

    save_csv(face_video_summary, "face_video_summary.csv")
    save_csv(failed_face_summary, "failed_face_summary.csv")

    print(
        f"\nSamples с низким face detection (<{MIN_VALID_FACE_RATIO}): "
        f"{len(failed_face_summary)} / {len(face_video_summary)} "
        f"({len(failed_face_summary)/len(face_video_summary)*100:.1f}%)"
    )

    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    # 1. Face detection ratio distribution
    for lab, c in [("real", "#2ecc71"), ("fake", "#e74c3c")]:
        s = face_video_summary[face_video_summary["label"] == lab]["face_detection_ratio"]
        if len(s) > 0:
            axes[0, 0].hist(s, bins=20, alpha=0.6, label=lab, color=c)
    axes[0, 0].set_title("Face detection ratio per sample")
    axes[0, 0].legend()
    axes[0, 0].axvline(MIN_VALID_FACE_RATIO, color="k", ls="--", lw=1)

    # 2. Mean face area ratio
    for lab, c in [("real", "#2ecc71"), ("fake", "#e74c3c")]:
        s = face_video_summary[face_video_summary["label"] == lab]["mean_face_area_ratio"]
        if len(s) > 0:
            axes[0, 1].hist(s, bins=25, alpha=0.6, label=lab, color=c)
    axes[0, 1].set_title("Mean face area ratio")
    axes[0, 1].legend()

    # 3. Faces per frame
    if len(df_faces) > 0:
        df_faces["n_faces"].value_counts().sort_index().plot.bar(ax=axes[0, 2], color="#3498db")
    axes[0, 2].set_title("Faces per frame")

    # 4. Face detection by dataset
    face_video_summary.groupby("source_dataset")["face_detection_ratio"].mean().plot.bar(
        ax=axes[1, 0], color="#9b59b6"
    )
    axes[1, 0].set_title("Mean face det. ratio by dataset")
    axes[1, 0].set_ylim(0, 1.1)

    # 5. Multi-face ratio by label
    face_video_summary.groupby("label")["multi_face_ratio"].mean().plot.bar(
        ax=axes[1, 1], color=["#e74c3c", "#2ecc71"]
    )
    axes[1, 1].set_title("Multi-face ratio by label")

    # 6. No-face ratio by label
    face_video_summary.groupby("label")["no_face_ratio"].mean().plot.bar(
        ax=axes[1, 2], color=["#e74c3c", "#2ecc71"]
    )
    axes[1, 2].set_title("Mean no-face ratio by label")
    axes[1, 2].set_ylim(0, 1.0)

    plt.suptitle("Face-level Analysis", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    save_fig("07_face_analysis")
    plt.show()

    mini_report("FACE ANALYSIS", {
        "Samples analyzed": len(face_video_summary),
        "Mean face det. ratio": f"{face_video_summary['face_detection_ratio'].mean():.3f}",
        "Samples failing face threshold": f"{len(failed_face_summary)} ({len(failed_face_summary)/len(face_video_summary)*100:.1f}%)",
        "Mean face area ratio": f"{face_video_summary['mean_face_area_ratio'].mean():.3f}",
        "Multi-face frames": f"{int(df_faces['multi_face'].sum())} / {len(df_faces)}" if len(df_faces) > 0 else "0 / 0",
    })
else:
    print("Face video summary is empty — skipping face summary plots.")

## 9. Quality Metrics Analysis

Blur, brightness, contrast, saturation, entropy — на кадрах и face crop'ах.
Проверяем: нет ли качественного bias между real и fake.

In [ ]:
# =====================================================================
# 9. QUALITY METRICS ANALYSIS
# =====================================================================

def compute_quality(image_rgb: np.ndarray) -> dict:
    """Метрики качества одного RGB-кадра / crop."""
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)

    hist = np.histogram(gray, bins=256, range=(0, 256))[0].astype(np.float64)
    prob = hist / max(hist.sum(), 1.0)
    entropy = -np.sum(prob * np.log2(prob + 1e-10))

    return {
        "blur_laplacian": round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 4),
        "brightness": round(float(gray.mean()), 4),
        "contrast": round(float(gray.std()), 4),
        "saturation": round(float(hsv[:, :, 1].mean()), 4),
        "entropy": round(float(entropy), 4),
    }

if RUN_HEAVY_ANALYSIS:
    print("Quality metrics на sampled frames...")

    quality_records = []
    q_sample = df[df["is_readable"]].sample(
        min(MAX_VIDEOS_FOR_HEAVY_ANALYSIS, len(df[df["is_readable"]])),
        random_state=RANDOM_SEED
    )

    for _, row in tqdm(q_sample.iterrows(), total=len(q_sample), desc="Frame quality"):
        if row["file_type"] == "video":
            frames = extract_frames_uniform(row["path"], n=min(3, MAX_FRAMES_PER_VIDEO_FOR_ANALYSIS))
        else:
            img = load_image_rgb(row["path"])
            frames = [img] if img is not None else []

        for fi, frame in enumerate(frames):
            if frame is None:
                continue

            qm = compute_quality(frame)
            qm.update({
                "sample_id": row["sample_id"],
                "frame_idx": fi,
                "path": row["path"],
                "label": row["label"],
                "source_dataset": row["source_dataset"],
                "height": int(frame.shape[0]),
                "width": int(frame.shape[1]),
            })
            quality_records.append(qm)

    df_quality = pd.DataFrame(quality_records)
    save_csv(df_quality, "frame_quality_metrics.csv")

    # Face crop quality
    face_quality_records = []

    if len(df_faces) > 0:
        print("Quality metrics на face crops...")

        # берем только реальные детекции лиц
        df_face_boxes = df_faces[df_faces["face_detected"] == True].copy()

        # ограничим объем
        face_box_sample = df_face_boxes.sample(
            min(300, len(df_face_boxes)),
            random_state=RANDOM_SEED
        ) if len(df_face_boxes) > 0 else pd.DataFrame()

        for _, row in tqdm(face_box_sample.iterrows(), total=len(face_box_sample), desc="Face crop quality"):
            img = load_image_rgb(row["path"])
            if img is None:
                continue

            x, y, w, h = row["x"], row["y"], row["w"], row["h"]
            if any(pd.isna(v) for v in [x, y, w, h]):
                continue

            x, y, w, h = int(x), int(y), int(w), int(h)
            if w <= 0 or h <= 0:
                continue

            crop = img[y:y+h, x:x+w]
            if crop.size == 0:
                continue

            qm = compute_quality(crop)
            qm.update({
                "sample_id": row["sample_id"],
                "frame_idx": row["frame_idx"],
                "path": row["path"],
                "label": row["label"],
                "source_dataset": row["source_dataset"],
                "face_w": w,
                "face_h": h,
                "face_area_ratio": row["face_area_ratio"],
            })
            face_quality_records.append(qm)

    df_face_quality = pd.DataFrame(face_quality_records)
    save_csv(df_face_quality, "face_quality_metrics.csv")

    print(f"Frame quality records: {len(df_quality)}")
    print(f"Face crop quality records: {len(df_face_quality)}")

else:
    df_quality = pd.DataFrame()
    df_face_quality = pd.DataFrame()
    print("Quality analysis skipped (RUN_HEAVY_ANALYSIS=False)")

In [ ]:
if len(df_quality) > 0:
    qmetrics = ['blur_laplacian', 'brightness', 'contrast', 'saturation', 'entropy']

    # KDE plots: real vs fake.
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes_flat = axes.flatten()

    for i, col in enumerate(qmetrics):
        ax = axes_flat[i]
        for lab, c in [('real','#2ecc71'),('fake','#e74c3c')]:
            s = df_quality[df_quality['label']==lab][col].dropna()
            if len(s) > 5:
                sns.kdeplot(s, ax=ax, label=lab, color=c, fill=True, alpha=0.3)
        ax.set_title(col)
        ax.legend()

    # Correlation heatmap.
    corr = df_quality[qmetrics].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes_flat[5],
                vmin=-1, vmax=1)
    axes_flat[5].set_title('Quality metrics correlation')

    plt.suptitle('Quality Metrics: Real vs Fake', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_fig("08_quality_metrics")
    plt.show()

    # Bias check: effect size.
    print("\n--- Quality Bias Check ---")
    for col in qmetrics:
        r = df_quality[df_quality['label']=='real'][col].dropna()
        f = df_quality[df_quality['label']=='fake'][col].dropna()
        if len(r) < 5 or len(f) < 5: continue
        d = abs(r.mean() - f.mean())
        pooled = np.sqrt((r.std()**2 + f.std()**2) / 2)
        es = d / pooled if pooled > 0 else 0
        flag = "⚠ BIAS RISK" if es > 0.5 else "✓ OK"
        print(f"  {col:20s}: real={r.mean():.1f} fake={f.mean():.1f} "
              f"effect_size={es:.2f} {flag}")

    print("\nЕсли effect_size > 0.5: модель может учиться на артефактах качества/источника, "
          "а не на deepfake-признаках.")

## 10. Temporal / Sequence-level Analysis

Ключевой раздел для ВКР: анализ пригодности данных для temporal branch (T=16).
Inter-frame differences, face persistence, usable clip lengths.

In [ ]:
# =====================================================================
# 10. TEMPORAL / SEQUENCE-LEVEL ANALYSIS
# Для твоих данных: sequence analysis по ПАПКАМ С КАДРАМИ
# =====================================================================

def load_frame_sequence_from_group(group_df: pd.DataFrame, max_len: int = TARGET_CLIP_LEN) -> List[np.ndarray]:
    """Загружает последовательность кадров из одной группы (group_id)."""
    if group_df.empty:
        return []

    seq_df = group_df.copy()
    seq_df["basename"] = seq_df["path"].apply(lambda x: Path(x).name)
    seq_df = seq_df.sort_values("basename")

    paths = seq_df["path"].tolist()[:max_len]

    frames = []
    for p in paths:
        img = load_image_rgb(p)
        if img is not None:
            frames.append(img)

    return frames


print("=== TEMPORAL / SEQUENCE-LEVEL ANALYSIS ===")

dfv_an = df[
    (df["file_type"] == "image") &
    (df["is_readable"]) &
    (df["source_dataset"] == PRIMARY_DATASET_NAME)
].copy()

print("Temporal source datasets:", dfv_an["source_dataset"].value_counts().to_dict())
print("Temporal frame rows found:", len(dfv_an))

temporal_records = []

if len(dfv_an) > 0 and RUN_HEAVY_ANALYSIS:
    group_sizes = (
        dfv_an.groupby("group_id")
        .size()
        .reset_index(name="n_frames_available")
        .sort_values("n_frames_available", ascending=False)
    )

    group_sizes = group_sizes[group_sizes["n_frames_available"] >= 2].copy()

    if len(group_sizes) > 0:
        sampled_groups = group_sizes.sample(
            min(MAX_VIDEOS_FOR_HEAVY_ANALYSIS, len(group_sizes)),
            random_state=RANDOM_SEED
        )["group_id"].tolist()

        for gid in tqdm(sampled_groups, desc="Temporal analysis"):
            group_df = dfv_an[dfv_an["group_id"] == gid].copy()
            if group_df.empty:
                continue

            frames = load_frame_sequence_from_group(group_df, max_len=TARGET_CLIP_LEN)
            if len(frames) < 2:
                continue

            diffs = []
            has_face_seq = []
            largest_face_seq = []

            for i in range(len(frames) - 1):
                d = np.abs(frames[i + 1].astype(np.float32) - frames[i].astype(np.float32)).mean()
                diffs.append(float(d))

            for f in frames:
                faces = detect_faces(f)
                has_face = len(faces) > 0
                has_face_seq.append(has_face)
                largest_face = max([x["area_ratio"] for x in faces], default=0.0)
                largest_face_seq.append(largest_face)

            face_persist = sum(has_face_seq) / len(has_face_seq)

            max_cont = 0
            curr = 0
            for hf in has_face_seq:
                if hf:
                    curr += 1
                    max_cont = max(max_cont, curr)
                else:
                    curr = 0

            temporal_records.append({
                "sample_id": group_df["sample_id"].iloc[0],
                "group_id": gid,
                "label": group_df["label"].mode().iloc[0] if not group_df["label"].mode().empty else "unknown",
                "source_dataset": group_df["source_dataset"].mode().iloc[0] if not group_df["source_dataset"].mode().empty else "unknown",
                "n_frames_available": int(len(group_df)),
                "n_frames_sampled": int(len(frames)),
                "mean_ifd": round(float(np.mean(diffs)), 4),
                "std_ifd": round(float(np.std(diffs)), 4),
                "max_ifd": round(float(np.max(diffs)), 4),
                "min_ifd": round(float(np.min(diffs)), 4),
                "face_persistence_ratio": round(float(face_persist), 4),
                "mean_face_area_ratio": round(float(np.mean(largest_face_seq)), 6),
                "max_continuous_valid_len": int(max_cont),
                "temporal_stability_score": round(float(1.0 / (1.0 + np.std(diffs))), 6),
                "usable_clip_count": int(1 if max_cont >= TARGET_CLIP_LEN // 2 else 0),
                "is_temporal_ready": bool(max_cont >= TARGET_CLIP_LEN // 2),
            })

df_temporal = pd.DataFrame(temporal_records)

if len(df_temporal) > 0:
    save_csv(df_temporal, "temporal_metrics.csv")
    clip_candidates = df_temporal[df_temporal["is_temporal_ready"]].copy()
    save_csv(clip_candidates, "clip_candidates.csv")
    print(f"Temporal metrics: {len(df_temporal)} sequences analyzed")
    print(f"Clip candidates: {len(clip_candidates)}")
else:
    clip_candidates = pd.DataFrame()
    print("Temporal analysis: нет данных для анализа")

In [ ]:
if len(df_temporal) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    # Mean IFD by label
    for lab, c in [("real", "#2ecc71"), ("fake", "#e74c3c")]:
        s = df_temporal[df_temporal["label"] == lab]["mean_ifd"].dropna()
        if len(s) > 0:
            axes[0, 0].hist(s, bins=20, alpha=0.6, label=lab, color=c)
    axes[0, 0].set_title("Mean inter-frame difference")
    axes[0, 0].legend()

    # Std IFD
    plot_df = df_temporal[df_temporal["label"].isin(["real", "fake"])].copy()
    if len(plot_df) > 0:
        sns.boxplot(
            data=plot_df,
            x="label",
            y="std_ifd",
            ax=axes[0, 1],
            palette={"real": "#2ecc71", "fake": "#e74c3c"},
        )
    axes[0, 1].set_title("IFD variability")

    # Face persistence
    for lab, c in [("real", "#2ecc71"), ("fake", "#e74c3c")]:
        s = df_temporal[df_temporal["label"] == lab]["face_persistence_ratio"].dropna()
        if len(s) > 0:
            axes[0, 2].hist(s, bins=15, alpha=0.6, label=lab, color=c)
    axes[0, 2].set_title("Face persistence ratio")
    axes[0, 2].legend()

    # Max continuous valid length
    df_temporal["max_continuous_valid_len"].hist(ax=axes[1, 0], bins=15, color="#3498db")
    axes[1, 0].axvline(TARGET_CLIP_LEN // 2, color="red", ls="--", label=f"T/2={TARGET_CLIP_LEN // 2}")
    axes[1, 0].set_title("Max continuous valid clip len")
    axes[1, 0].legend()

    # Temporal readiness
    tr = df_temporal["is_temporal_ready"].value_counts()
    tr.plot.bar(ax=axes[1, 1], color=["#e74c3c", "#2ecc71"])
    axes[1, 1].set_title("Temporal readiness")

    # Mean IFD real vs fake
    if len(plot_df) > 0:
        sns.boxplot(
            data=plot_df,
            x="label",
            y="mean_ifd",
            ax=axes[1, 2],
            palette={"real": "#2ecc71", "fake": "#e74c3c"},
        )
    axes[1, 2].set_title("Mean IFD: real vs fake")

    plt.suptitle("Temporal / Sequence Analysis", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    save_fig("09_temporal_analysis")
    plt.show()

    clip_candidates = df_temporal[df_temporal["is_temporal_ready"]].copy()
    save_csv(clip_candidates, "clip_candidates.csv")

    real_ifd_series = df_temporal[df_temporal["label"] == "real"]["mean_ifd"].dropna()
    fake_ifd_series = df_temporal[df_temporal["label"] == "fake"]["mean_ifd"].dropna()

    mini_report("TEMPORAL ANALYSIS", {
        "Sequences analyzed": len(df_temporal),
        "Temporal-ready": f"{int(df_temporal['is_temporal_ready'].sum())} ({df_temporal['is_temporal_ready'].mean()*100:.1f}%)",
        "Mean face persistence": f"{df_temporal['face_persistence_ratio'].mean():.3f}",
        "Mean IFD (real)": f"{real_ifd_series.mean():.4f}" if len(real_ifd_series) > 0 else "N/A",
        "Mean IFD (fake)": f"{fake_ifd_series.mean():.4f}" if len(fake_ifd_series) > 0 else "N/A",
        "Clip candidates": len(clip_candidates),
    })

    if len(real_ifd_series) > 0 and len(fake_ifd_series) > 0:
        real_ifd = real_ifd_series.mean()
        fake_ifd = fake_ifd_series.mean()

        if abs(real_ifd - fake_ifd) / max(real_ifd, fake_ifd, 0.01) > 0.1:
            print("→ Заметная разница IFD между real и fake: это поддерживает гипотезу о полезности temporal features.")
        else:
            print("→ Разница IFD невелика на уровне среднего; модели нужны более тонкие temporal паттерны.")

## 11. Formal Cleaning Rules

Определяем machine-readable правила очистки. Применяем их. Сохраняем dropped_samples.csv с причиной удаления.

In [ ]:
CLEANING_RULES = OrderedDict([
    ("unreadable",       lambda row: not row["is_readable"]),
    ("empty_file",       lambda row: row["size_bytes"] < 1024),
    ("missing_label",    lambda row: row["label"] == "unknown"),
    ("low_resolution",   lambda row: row["is_readable"] and min(row["width"], row["height"]) < MIN_RESOLUTION),
    ("too_short_video",  lambda row: row["file_type"] == "video" and row["duration_sec"] < MIN_DURATION_SEC),
    ("invalid_fps",      lambda row: row["file_type"] == "video" and (
        row["fps"] <= 0 or row["fps"] < VALID_FPS_RANGE[0] or row["fps"] > VALID_FPS_RANGE[1])),
    ("too_few_frames",   lambda row: row["file_type"] == "video" and row["frame_count"] < TARGET_CLIP_LEN),
    ("duplicate_hash",   lambda row: "duplicate_hash" in str(row.get("issue_flags", ""))),
])

print("=== CLEANING RULES ===")
for name in CLEANING_RULES.keys():
    print(f"  • {name}")

dropped_records = []
keep_mask = pd.Series(True, index=df.index)

for rule_name, rule_fn in CLEANING_RULES.items():
    rule_mask = df.apply(rule_fn, axis=1)
    newly_dropped = rule_mask & keep_mask
    n_dropped = int(newly_dropped.sum())

    if n_dropped > 0:
        dropped_chunk = df.loc[newly_dropped, ["sample_id", "path", "source_dataset", "label"]].copy()
        dropped_chunk["drop_stage"] = "cleaning"
        dropped_chunk["drop_reason"] = rule_name
        dropped_chunk["drop_reason_group"] = rule_name.split("_")[0]
        dropped_records.append(dropped_chunk)

    keep_mask = keep_mask & ~rule_mask
    print(f"  {rule_name}: удалено {n_dropped}, осталось {int(keep_mask.sum())}")

df_dropped = pd.concat(dropped_records, ignore_index=True) if dropped_records else pd.DataFrame(
    columns=["sample_id", "path", "source_dataset", "label", "drop_stage", "drop_reason", "drop_reason_group"]
)
df_clean = df.loc[keep_mask].copy()

save_csv(df_dropped, "dropped_samples.csv", CLEAN_DIR)
save_csv(df_clean, "cleaned_metadata.csv", CLEAN_DIR)

print(f"\nИтого: {len(df)} → {len(df_clean)} (удалено {len(df_dropped)})")

In [ ]:
# Визуализация очистки.
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Причины удаления.
if len(df_dropped) > 0:
    df_dropped['drop_reason'].value_counts().plot.barh(ax=axes[0], color='#e74c3c')
    axes[0].set_title('Причины удаления')
    axes[0].invert_yaxis()
else:
    axes[0].text(0.5, 0.5, 'Ничего не удалено', transform=axes[0].transAxes, ha='center')
    axes[0].set_title('Причины удаления')

# 2. До/после по классам.
before = df['label'].value_counts()
after = df_clean['label'].value_counts()
comp = pd.DataFrame({'before': before, 'after': after}).fillna(0)
comp.plot.bar(ax=axes[1], color=['#95a5a6', '#2ecc71'])
axes[1].set_title('Классы: до / после очистки')

# 3. До/после по датасетам.
before_ds = df['source_dataset'].value_counts()
after_ds = df_clean['source_dataset'].value_counts()
comp_ds = pd.DataFrame({'before': before_ds, 'after': after_ds}).fillna(0)
comp_ds.plot.bar(ax=axes[2], color=['#95a5a6', '#3498db'])
axes[2].set_title('Датасеты: до / после очистки')

plt.tight_layout()
save_fig("10_cleaning_effect")
plt.show()

# Raw vs cleaned comparison table.
raw_counts = df.groupby(['source_dataset','label']).size().unstack(fill_value=0)
clean_counts = df_clean.groupby(['source_dataset','label']).size().unstack(fill_value=0)
comparison = pd.concat([raw_counts.add_prefix('raw_'), clean_counts.add_prefix('clean_')], axis=1).fillna(0).astype(int)
save_csv(comparison.reset_index(), "raw_vs_cleaned_comparison.csv", CLEAN_DIR)
print(comparison)

## 12. Cleaned Dataset Summary

Итоговое состояние данных после очистки.

In [ ]:
print("=== CLEANED DATASET SUMMARY ===\n")

for ds in df_clean['source_dataset'].unique():
    sub = df_clean[df_clean['source_dataset']==ds]
    print(f"--- {ds} ---")
    print(f"  Total: {len(sub)}")
    print(f"  Real: {len(sub[sub['label']=='real'])}")
    print(f"  Fake: {len(sub[sub['label']=='fake'])}")
    if len(sub) > 0 and 'real' in sub['label'].values and 'fake' in sub['label'].values:
        r = len(sub[sub['label']=='fake']) / max(len(sub[sub['label']=='real']),1)
        print(f"  Fake/Real ratio: {r:.2f}")
    print()

# Temporal readiness (только для видео).
vid_clean = df_clean[df_clean['file_type']=='video']
if len(vid_clean) > 0:
    if len(df_temporal) > 0:
        temporal_ids = set(df_temporal[df_temporal['is_temporal_ready']]['sample_id'])
        n_temporal_ready = vid_clean['sample_id'].isin(temporal_ids).sum()
        print(f"Temporal-ready videos (in cleaned set): {n_temporal_ready} / {len(vid_clean)}")

# Cleaned summary table.
summary_rows = []
for ds in df_clean['source_dataset'].unique():
    sub = df_clean[df_clean['source_dataset']==ds]
    summary_rows.append({
        'dataset': ds,
        'total': len(sub),
        'real': len(sub[sub['label']=='real']),
        'fake': len(sub[sub['label']=='fake']),
        'images': len(sub[sub['file_type']=='image']),
        'videos': len(sub[sub['file_type']=='video']),
    })
df_clean_summary = pd.DataFrame(summary_rows)
save_csv(df_clean_summary, "cleaned_summary_tables.csv", CLEAN_DIR)
print(df_clean_summary.to_string(index=False))

## 13. Preprocessing Pipeline & Train/Val/Test Manifests

Генерируем manifest-файлы для обучения. Split по group_id (leakage-safe). Stratified по label.

In [ ]:
# === Preprocessing config ===
PREPROCESS_CONFIG = {
    'clip_length': TARGET_CLIP_LEN,
    'spatial_size': 224,
    'temporal_size': 128,
    'face_crop': 'MTCNN (facenet-pytorch)',
    'spatial_norm': 'ImageNet mean/std',
    'temporal_input': 'frame differences (T-1)',
    'temporal_norm': 'zero-center + unit variance per clip',
    'sampling': 'uniform',
    'augmentation': 'HFlip, brightness±0.1, contrast±0.1 (train only)',
}

print("=== PREPROCESSING CONFIG ===")
for k, v in PREPROCESS_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# === Split generation ===
# Используем DFD_Frames для train/val/test.
# DFDC_Video — отдельно для raw video audit / auxiliary evaluation.

df_for_split = df_clean[df_clean["source_dataset"] == PRIMARY_DATASET_NAME].copy()
df_cross = df_clean[df_clean["source_dataset"] != PRIMARY_DATASET_NAME].copy()

print(f"Для split: {len(df_for_split)} samples ({PRIMARY_DATASET_NAME})")
print(f"Для cross/raw-video eval: {len(df_cross)} samples (secondary)")

if len(df_for_split) == 0:
    print("[ОШИБКА] Нет данных для split!")
else:
    rng = np.random.RandomState(RANDOM_SEED)

    group_labels = (
        df_for_split.groupby("group_id")["label"]
        .first()
        .reset_index()
    )

    manifests = {"train": [], "val": [], "test": []}

    for label_val in ["real", "fake"]:
        groups = group_labels[group_labels["label"] == label_val]["group_id"].values.copy()
        rng.shuffle(groups)

        n = len(groups)
        n_train = int(n * 0.70)
        n_val = int(n * 0.15)

        train_groups = set(groups[:n_train])
        val_groups = set(groups[n_train:n_train + n_val])
        test_groups = set(groups[n_train + n_val:])

        manifests["train"].extend(df_for_split[df_for_split["group_id"].isin(train_groups)].index.tolist())
        manifests["val"].extend(df_for_split[df_for_split["group_id"].isin(val_groups)].index.tolist())
        manifests["test"].extend(df_for_split[df_for_split["group_id"].isin(test_groups)].index.tolist())

    if len(face_video_summary) > 0:
        face_dict = face_video_summary.set_index("sample_id")[
            ["face_detection_ratio", "mean_face_area_ratio", "multi_face_ratio", "no_face_ratio"]
        ].to_dict("index")

        df_for_split["face_detection_ratio"] = df_for_split["sample_id"].map(
            lambda x: face_dict.get(x, {}).get("face_detection_ratio", np.nan)
        )
        df_for_split["mean_face_area_ratio"] = df_for_split["sample_id"].map(
            lambda x: face_dict.get(x, {}).get("mean_face_area_ratio", np.nan)
        )
        df_for_split["multi_face_ratio"] = df_for_split["sample_id"].map(
            lambda x: face_dict.get(x, {}).get("multi_face_ratio", np.nan)
        )
        df_for_split["no_face_ratio"] = df_for_split["sample_id"].map(
            lambda x: face_dict.get(x, {}).get("no_face_ratio", np.nan)
        )

    if len(df_temporal) > 0:
        temporal_dict = df_temporal.set_index("group_id")[
            ["is_temporal_ready", "face_persistence_ratio", "max_continuous_valid_len", "mean_ifd"]
        ].to_dict("index")

        df_for_split["is_temporal_ready"] = df_for_split["group_id"].map(
            lambda x: temporal_dict.get(x, {}).get("is_temporal_ready", np.nan)
        )
        df_for_split["face_persistence_ratio"] = df_for_split["group_id"].map(
            lambda x: temporal_dict.get(x, {}).get("face_persistence_ratio", np.nan)
        )
        df_for_split["max_continuous_valid_len"] = df_for_split["group_id"].map(
            lambda x: temporal_dict.get(x, {}).get("max_continuous_valid_len", np.nan)
        )
        df_for_split["mean_ifd"] = df_for_split["group_id"].map(
            lambda x: temporal_dict.get(x, {}).get("mean_ifd", np.nan)
        )

    manifest_cols = [
        "sample_id",
        "path",
        "label",
        "label_int",
        "source_dataset",
        "group_id",
        "file_type",
        "width",
        "height",
        "fps",
        "duration_sec",
        "aspect_ratio",
        "face_detection_ratio",
        "mean_face_area_ratio",
        "multi_face_ratio",
        "no_face_ratio",
        "is_temporal_ready",
        "face_persistence_ratio",
        "max_continuous_valid_len",
        "mean_ifd",
    ]

    split_dfs = {}

    for split_name, indices in manifests.items():
        m = df_for_split.loc[indices].copy()
        m["group_id_for_split"] = m["group_id"]
        m["split"] = split_name

        available_cols = [c for c in manifest_cols if c in m.columns]
        m = m[available_cols + ["group_id_for_split", "split"]]

        save_csv(m, f"{split_name}_manifest.csv", MANIFESTS_DIR)
        split_dfs[split_name] = m

        print(
            f"  {split_name}: {len(m)} samples "
            f"(real={len(m[m['label']=='real'])}, fake={len(m[m['label']=='fake'])})"
        )

    train_ids = set(split_dfs["train"]["sample_id"]) if "train" in split_dfs else set()
    val_ids = set(split_dfs["val"]["sample_id"]) if "val" in split_dfs else set()
    test_ids = set(split_dfs["test"]["sample_id"]) if "test" in split_dfs else set()

    assert train_ids.isdisjoint(val_ids), "train/val overlap detected"
    assert train_ids.isdisjoint(test_ids), "train/test overlap detected"
    assert val_ids.isdisjoint(test_ids), "val/test overlap detected"

    train_groups = set(split_dfs["train"]["group_id_for_split"]) if "train" in split_dfs else set()
    val_groups = set(split_dfs["val"]["group_id_for_split"]) if "val" in split_dfs else set()
    test_groups = set(split_dfs["test"]["group_id_for_split"]) if "test" in split_dfs else set()

    assert train_groups.isdisjoint(val_groups), "train/val group leakage detected"
    assert train_groups.isdisjoint(test_groups), "train/test group leakage detected"
    assert val_groups.isdisjoint(test_groups), "val/test group leakage detected"

    print("Split sanity checks: OK")

    if len(df_cross) > 0:
        df_cross_m = df_cross.copy()
        df_cross_m["group_id_for_split"] = df_cross_m["group_id"]
        df_cross_m["split"] = "cross_eval"

        available_cols = [c for c in manifest_cols if c in df_cross_m.columns]
        df_cross_m = df_cross_m[available_cols + ["group_id_for_split", "split"]]

        save_csv(df_cross_m, "cross_eval_manifest.csv", MANIFESTS_DIR)
        print(f"  cross_eval: {len(df_cross_m)} samples")

In [ ]:
# === Leakage sanity check ===
print("\n=== LEAKAGE SANITY CHECK ===")

train_m = pd.read_csv(Path(MANIFESTS_DIR) / "train_manifest.csv")
val_m   = pd.read_csv(Path(MANIFESTS_DIR) / "val_manifest.csv")
test_m  = pd.read_csv(Path(MANIFESTS_DIR) / "test_manifest.csv")

train_groups = set(train_m["group_id_for_split"])
val_groups   = set(val_m["group_id_for_split"])
test_groups  = set(test_m["group_id_for_split"])

tv_overlap = train_groups & val_groups
tt_overlap = train_groups & test_groups
vt_overlap = val_groups & test_groups

print(f"  train ∩ val:  {len(tv_overlap)} groups {'✓ OK' if len(tv_overlap) == 0 else '⚠ LEAKAGE!'}")
print(f"  train ∩ test: {len(tt_overlap)} groups {'✓ OK' if len(tt_overlap) == 0 else '⚠ LEAKAGE!'}")
print(f"  val ∩ test:   {len(vt_overlap)} groups {'✓ OK' if len(vt_overlap) == 0 else '⚠ LEAKAGE!'}")

if len(tv_overlap) + len(tt_overlap) + len(vt_overlap) == 0:
    print("  ✓ NO LEAKAGE DETECTED")
else:
    print("  ⚠ LEAKAGE DETECTED — FIX SPLIT LOGIC")

## 14. Post-preprocessing EDA

Финальная проверка train/val/test split: баланс, распределения, sanity checks.

In [ ]:
# Объединяем все manifests.
all_manifests = pd.concat([train_m, val_m, test_m], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Split × label.
ct = pd.crosstab(all_manifests['split'], all_manifests['label'])
ct.plot.bar(ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Split × Label')
axes[0].tick_params(axis='x', rotation=0)

# 2. Split sizes.
all_manifests['split'].value_counts().plot.bar(ax=axes[1], color='#3498db')
axes[1].set_title('Split sizes')
axes[1].tick_params(axis='x', rotation=0)

# 3. Class balance per split.
balance = all_manifests.groupby('split')['label'].value_counts(normalize=True).unstack()
balance.plot.bar(ax=axes[2], color=['#e74c3c', '#2ecc71'])
axes[2].set_title('Class proportion per split')
axes[2].set_ylabel('Proportion')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
save_fig("11_split_eda")
plt.show()

# Split summary table.
split_summary = all_manifests.groupby('split').agg(
    total=('sample_id', 'count'),
    real=('label', lambda x: (x=='real').sum()),
    fake=('label', lambda x: (x=='fake').sum()),
).reset_index()
split_summary['fake_ratio'] = (split_summary['fake'] / split_summary['total']).round(3)
save_csv(split_summary, "split_summary.csv", MANIFESTS_DIR)
print(split_summary.to_string(index=False))

# Balance table.
balance_tbl = pd.crosstab(all_manifests['split'], all_manifests['label'], margins=True)
save_csv(balance_tbl.reset_index(), "split_balance_tables.csv", MANIFESTS_DIR)
print(f"\n{balance_tbl}")

## 15. Final Analytical Report

In [ ]:
print("=" * 70)
print("  ФИНАЛЬНЫЙ АНАЛИТИЧЕСКИЙ ОТЧЁТ")
print("=" * 70)

total_raw = len(df)
total_clean = len(df_clean)
total_dropped = len(df_dropped)
n_videos_raw = int((df["file_type"] == "video").sum())
n_images_raw = int((df["file_type"] == "image").sum())
n_issues = int(df["has_issues"].sum()) if "has_issues" in df.columns else 0

issues_pct = (n_issues / total_raw * 100) if total_raw > 0 else 0.0
dropped_pct = (total_dropped / total_raw * 100) if total_raw > 0 else 0.0

report = f"""
1. RAW DATA
   Всего samples: {total_raw}
   Изображений: {n_images_raw} | Видео: {n_videos_raw}
   С проблемами: {n_issues} ({issues_pct:.1f}%)

2. QUALITY ISSUES
   Удалено при очистке: {total_dropped} ({dropped_pct:.1f}%)
"""

if len(df_dropped) > 0:
    top_reasons = df_dropped["drop_reason"].value_counts().head(5)
    for reason, cnt in top_reasons.items():
        report += f"   • {reason}: {cnt}\n"

report += f"""
3. LEAKAGE
   Split key: group_id
   Leakage between splits: {'NONE ✓' if len(tv_overlap) + len(tt_overlap) + len(vt_overlap) == 0 else 'DETECTED ⚠'}
   Датасеты НЕ смешаны: DFD_Frames=train/val/test, DFDC_Video=raw/cross-eval

4. FACE ANALYSIS
"""

if len(face_video_summary) > 0:
    failed_face_summary = face_video_summary[
        face_video_summary["face_detection_ratio"] < MIN_VALID_FACE_RATIO
    ].copy()
    report += f"   Face detection rate (mean): {face_video_summary['face_detection_ratio'].mean():.3f}\n"
    report += f"   Samples failing face threshold: {len(failed_face_summary)}\n"
else:
    report += "   (пропущен — RUN_HEAVY_ANALYSIS=False или face summary empty)\n"

report += f"""
5. TEMPORAL READINESS
"""

if len(df_temporal) > 0:
    report += f"   Temporal-ready sequences: {int(df_temporal['is_temporal_ready'].sum())} / {len(df_temporal)}\n"
    report += f"   Mean face persistence: {df_temporal['face_persistence_ratio'].mean():.3f}\n"
else:
    report += "   (пропущен — нет последовательностей или RUN_HEAVY_ANALYSIS=False)\n"

report += f"""
6. CLEANED DATASET
   Total clean samples: {total_clean}
"""

for ds in df_clean["source_dataset"].unique():
    sub = df_clean[df_clean["source_dataset"] == ds]
    report += (
        f"   {ds}: {len(sub)} "
        f"(real={int((sub['label'] == 'real').sum())}, fake={int((sub['label'] == 'fake').sum())})\n"
    )

report += f"""
7. TRAIN/VAL/TEST SPLIT
   Train: {len(train_m)} | Val: {len(val_m)} | Test: {len(test_m)}

8. ОСТАВШИЕСЯ ОГРАНИЧЕНИЯ
   • Haar Cascade менее точен, чем MTCNN — реальный face det. rate может быть выше
   • Quality bias check выполнен на подвыборке — результат может отличаться на полных данных
   • Temporal analysis выполнен по последовательностям кадров, а не по исходным видео напрямую

9. РЕКОМЕНДАЦИИ ДЛЯ МОДЕЛИРОВАНИЯ
   → Основное обучение: DFD_Frames (manifests в {MANIFESTS_DIR}/)
   → Raw video audit / auxiliary eval: DFDC_Video
   → Baseline: spatial-only, затем full dual-path model
   → Ablation: spatial-only vs full vs sequential temporal model
"""

print(report)

report_path = Path(REPORTS_DIR) / "final_analytical_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(f"Отчёт сохранён: {report_path}")

## 16. Checklist Before Modeling

| # | Пункт | Статус |
|---|-------|--------|
| 1 | Raw inventory completed | ✅ `master_raw_inventory.csv` |
| 2 | Quality issues registered | ✅ `problem_registry.csv` |
| 3 | Duplicates checked (hash + name) | ✅ |
| 4 | Leakage groups identified | ✅ `leakage_groups.csv` |
| 5 | Face detection coverage measured | ✅ `face_video_summary.csv` |
| 6 | Frame quality metrics computed | ✅ `frame_quality_metrics.csv` |
| 7 | Temporal readiness measured | ✅ `temporal_metrics.csv` |
| 8 | Cleaning rules applied | ✅ `dropped_samples.csv` |
| 9 | Cleaned metadata saved | ✅ `cleaned_metadata.csv` |
| 10 | Train/val/test manifests saved | ✅ `train/val/test_manifest.csv` |
| 11 | Split leakage sanity check passed | ✅ |
| 12 | Dataset ready for baseline training | ✅ |

**Все артефакты сохранены в `reports/`.**

**Следующий шаг:** запуск `train.py` с указанием на manifests.

In [ ]:
# Финальная проверка: все артефакты на месте
print("=== ARTIFACT CHECK ===")

has_video_files = ((df["file_type"] == "video").sum() > 0)

expected = [
    ("tables", "master_raw_inventory.csv"),
    ("tables", "inventory_errors.csv"),
    ("tables", "unsupported_files.csv"),
    ("tables", "dataset_label_crosstab.csv"),
    ("tables", "summary_video_stats.csv"),
    ("tables", "summary_image_stats.csv"),
    ("tables", "problem_registry.csv"),
    ("tables", "quality_issue_summary.csv"),
    ("tables", "leakage_candidates.csv"),
    ("tables", "leakage_groups.csv"),
    ("tables", "frame_samples_metadata.csv"),
    ("tables", "face_detections.csv"),
    ("tables", "face_video_summary.csv"),
    ("tables", "failed_face_cases.csv"),
    ("tables", "frame_quality_metrics.csv"),
    ("tables", "face_quality_metrics.csv"),
    ("cleaned", "dropped_samples.csv"),
    ("cleaned", "cleaned_metadata.csv"),
    ("cleaned", "cleaned_summary_tables.csv"),
    ("cleaned", "raw_vs_cleaned_comparison.csv"),
    ("manifests", "train_manifest.csv"),
    ("manifests", "val_manifest.csv"),
    ("manifests", "test_manifest.csv"),
    ("manifests", "split_summary.csv"),
    ("manifests", "split_balance_tables.csv"),
]

optional_expected = [
    ("manifests", "cross_eval_manifest.csv"),
    ("cleaned", "failed_face_summary.csv"),
]

if has_video_files:
    expected += [
        ("tables", "temporal_metrics.csv"),
        ("tables", "clip_candidates.csv"),
    ]
else:
    optional_expected += [
        ("tables", "temporal_metrics.csv"),
        ("tables", "clip_candidates.csv"),
    ]

dirs_map = {
    "tables": TABLES_DIR,
    "cleaned": CLEAN_DIR,
    "manifests": MANIFESTS_DIR,
}

found = 0
missing = 0

for subdir, fname in expected:
    fpath = Path(dirs_map[subdir]) / fname
    exists = fpath.is_file()
    status = "✅" if exists else "❌"
    if exists:
        found += 1
    else:
        missing += 1
    print(f"  {status} {subdir}/{fname}")

print("\n--- Optional artifacts ---")
for subdir, fname in optional_expected:
    fpath = Path(dirs_map[subdir]) / fname
    exists = fpath.is_file()
    status = "✅" if exists else "—"
    print(f"  {status} {subdir}/{fname}")

print(f"\nVideo files present in inventory: {has_video_files}")
print(f"Найдено: {found}/{len(expected)} обязательных артефактов")

if missing > 0:
    print(f"  ⚠ Отсутствует {missing} обязательных артефактов")
else:
    print("  ✓ Все обязательные артефакты на месте")

print("\n=== NOTEBOOK ЗАВЕРШЁН ===")

In [ ]:
print(df["file_type"].value_counts(dropna=False))
print(df["source_dataset"].value_counts(dropna=False))
print(df[df["source_dataset"].str.contains("DFD", case=False, na=False)][["path", "file_type"]].head(20))

In [ ]:
print("PRIMARY_DATASET_ROOT =", PRIMARY_DATASET_ROOT)
print("SECONDARY_DATASET_ROOT =", SECONDARY_DATASET_ROOT)
print("PRIMARY_DATASET_NAME =", PRIMARY_DATASET_NAME)

from pathlib import Path

primary_files = list(Path(PRIMARY_DATASET_ROOT).rglob("*"))
primary_files = [p for p in primary_files if p.is_file()][:20]

print(f"\n--- PRIMARY dataset sample files ({len(primary_files)} shown) ---")
for p in primary_files[:10]:
    print(p)

if HAS_SECONDARY:
    secondary_files = list(Path(SECONDARY_DATASET_ROOT).rglob("*"))
    secondary_files = [p for p in secondary_files if p.is_file()][:20]
    print(f"\n--- SECONDARY dataset sample files ({len(secondary_files)} shown) ---")
    for p in secondary_files[:10]:
        print(p)


In [ ]:
# =====================================================================
# EXPORT ONE DOCUMENT (SAVE INTO PROJECT ROOT) + PDF
# =====================================================================

import os
import json
import base64
import html
from pathlib import Path
from datetime import datetime

# Сохраняем РЯДОМ С ПРОЕКТОМ
EXPORT_DIR = REPORTS_DIR
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

HTML_REPORT_PATH = EXPORT_DIR / "FULL_EDA_REPORT.html"
PDF_REPORT_PATH = EXPORT_DIR / "FULL_EDA_REPORT.pdf"

def img_to_base64(path: Path) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def safe_read_text(path: Path, max_chars: int = 120000) -> str:
    if not path.exists():
        return ""
    txt = path.read_text(encoding="utf-8", errors="ignore")
    if len(txt) > max_chars:
        txt = txt[:max_chars] + "\n\n...[truncated]..."
    return txt

generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

summary = {
    "generated_at": generated_at,
    "project_root": str(PROJECT_ROOT),
    "reports_dir": str(REPORTS_DIR),
    "total_raw": int(len(df)) if "df" in globals() else 0,
    "total_clean": int(len(df_clean)) if "df_clean" in globals() else 0,
    "total_dropped": int(len(df_dropped)) if "df_dropped" in globals() else 0,
    "images_raw": int((df["file_type"] == "image").sum()) if "df" in globals() and "file_type" in df.columns else 0,
    "videos_raw": int((df["file_type"] == "video").sum()) if "df" in globals() and "file_type" in df.columns else 0,
    "source_dataset_counts": df["source_dataset"].value_counts().to_dict() if "df" in globals() and "source_dataset" in df.columns else {},
    "file_type_counts": df["file_type"].value_counts().to_dict() if "df" in globals() and "file_type" in df.columns else {},
    "split_sizes": {
        "train": int(len(train_m)) if "train_m" in globals() else 0,
        "val": int(len(val_m)) if "val_m" in globals() else 0,
        "test": int(len(test_m)) if "test_m" in globals() else 0,
    },
    "temporal_sequences": int(len(df_temporal)) if "df_temporal" in globals() else 0,
    "clip_candidates": int(len(clip_candidates)) if "clip_candidates" in globals() else 0,
}

final_report_text = safe_read_text(Path(REPORTS_DIR) / "final_analytical_report.txt")
plot_files = sorted([p for p in Path(PLOTS_DIR).glob("*.png") if p.is_file()])

# -----------------------------
# HTML
# -----------------------------
html_parts = []
html_parts.append(f"""
<!DOCTYPE html>
<html lang="ru">
<head>
<meta charset="UTF-8">
<title>FULL EDA REPORT</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 24px;
    line-height: 1.45;
    color: #222;
}}
h1, h2, h3 {{
    margin-top: 28px;
}}
pre {{
    background: #f5f5f5;
    padding: 12px;
    border-radius: 8px;
    overflow-x: auto;
    white-space: pre-wrap;
}}
img {{
    max-width: 100%;
    border: 1px solid #ddd;
    border-radius: 6px;
    margin: 10px 0 24px 0;
}}
</style>
</head>
<body>
<h1>FULL EDA REPORT</h1>
<p><b>Generated:</b> {html.escape(generated_at)}</p>
<p><b>Project root:</b> {html.escape(str(PROJECT_ROOT))}</p>

<h2>1. Run Summary</h2>
<pre>{html.escape(json.dumps(summary, ensure_ascii=False, indent=2))}</pre>

<h2>2. Final Analytical Report</h2>
<pre>{html.escape(final_report_text if final_report_text else "No final analytical report found")}</pre>

<h2>3. Plots</h2>
""")

if plot_files:
    for p in plot_files:
        b64 = img_to_base64(p)
        html_parts.append(f"""
        <h3>{html.escape(p.name)}</h3>
        <img src="data:image/png;base64,{b64}" alt="{html.escape(p.name)}">
        """)
else:
    html_parts.append("<p>No plots found.</p>")

html_parts.append("</body></html>")
HTML_REPORT_PATH.write_text("".join(html_parts), encoding="utf-8")

# -----------------------------
# PDF
# -----------------------------
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Image as RLImage, Preformatted
from reportlab.lib.enums import TA_LEFT
from reportlab.lib import colors

pdf_doc = SimpleDocTemplate(
    str(PDF_REPORT_PATH),
    pagesize=A4,
    rightMargin=18*mm,
    leftMargin=18*mm,
    topMargin=16*mm,
    bottomMargin=16*mm,
)

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    name="BodySmall",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=9.5,
    leading=13,
    alignment=TA_LEFT,
    spaceAfter=6,
))
styles.add(ParagraphStyle(
    name="TitleBig",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=18,
    leading=22,
    textColor=colors.black,
    spaceAfter=10,
))
styles.add(ParagraphStyle(
    name="SectionHead",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=13,
    leading=16,
    textColor=colors.black,
    spaceBefore=10,
    spaceAfter=8,
))
styles.add(ParagraphStyle(
    name="CodeBlock",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=8,
    leading=10,
    backColor=colors.whitesmoke,
    borderPadding=6,
    spaceAfter=8,
))

story = []
story.append(Paragraph("FULL EDA REPORT", styles["TitleBig"]))
story.append(Paragraph(f"Generated: {generated_at}", styles["BodySmall"]))
story.append(Paragraph(f"Project root: {str(PROJECT_ROOT)}", styles["BodySmall"]))
story.append(Spacer(1, 8))

story.append(Paragraph("1. Run Summary", styles["SectionHead"]))
story.append(Preformatted(json.dumps(summary, ensure_ascii=False, indent=2), styles["CodeBlock"]))

story.append(Paragraph("2. Final Analytical Report", styles["SectionHead"]))
story.append(Preformatted(final_report_text if final_report_text else "No final analytical report found", styles["CodeBlock"]))

story.append(PageBreak())
story.append(Paragraph("3. Plots", styles["SectionHead"]))

page_width = A4[0] - pdf_doc.leftMargin - pdf_doc.rightMargin
max_img_w = page_width
max_img_h = A4[1] - pdf_doc.topMargin - pdf_doc.bottomMargin - 35*mm

for p in plot_files:
    story.append(Paragraph(p.name, styles["BodySmall"]))

    img = RLImage(str(p))
    iw, ih = img.drawWidth, img.drawHeight
    scale = min(max_img_w / iw, max_img_h / ih, 1.0)
    img.drawWidth = iw * scale
    img.drawHeight = ih * scale

    story.append(img)
    story.append(Spacer(1, 8))

pdf_doc.build(story)

print("DONE")
print("HTML:", HTML_REPORT_PATH)
print("PDF :", PDF_REPORT_PATH)